# New uncertainty workflow
This notebook mirrors the new-data workflow: select a subset of shoreline shapefiles, pull `CPS` from each shoreline shapefile's attribute table, resolve `Pixel_ER` and `Photoscale` from source metadata, derive `Georef_ER`, and compute `Total_UNCY = sqrt(Ep^2 + Eg^2 + Ed^2)`.

`Photoscale` is not part of the final equation directly; it is used to determine `Georef_ER` for source types that need it.

In [ ]:
%load_ext autotime
import geopandas as gpd
import pandas as pd
import numpy as np
import math
import re
from pathlib import Path
from glob import glob
from tqdm.auto import tqdm
import platform
import rasterio
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 130)

if platform.system() == 'Windows':
    prefix = r'Z:/'
else:
    prefix = 'ressci201900060-RNC2-Coastal/'

# EDIT TARGET SHORELINES HERE
# See Part 6 of GETTING_STARTED.md; use the same RUN_OWNER and search criteria in all 3 notebooks.
cutoff_date = pd.Timestamp('2024-07-18')
search_roots = [Path(r'Z:\MaxarImagery\HighFreq'), Path(r'Z:\Retrolens')]
search_mode = 'region_in_date_range'  # 'date', 'aoi', 'aoi_in_date_range', 'region', or 'region_in_date_range'
target_aoi = 'MedlandsBeach'
target_region = 'Auckland'

# Put your own name here. All three notebooks must use the same value, so your outputs
# stay in your own folder and can never overwrite someone else's run of the same area.
RUN_OWNER = 'yourname'
DATA_DIR = Path('DataUpdatev2') / RUN_OWNER
DATA_DIR.mkdir(parents=True, exist_ok=True)

def _norm(s):
    return ''.join(ch for ch in str(s).lower() if ch.isalnum())

valid_modes = {'date', 'aoi', 'aoi_in_date_range', 'region', 'region_in_date_range'}
if search_mode not in valid_modes:
    raise ValueError(f'search_mode must be one of {sorted(valid_modes)}')

if search_mode in {'aoi', 'aoi_in_date_range'} and not str(target_aoi).strip():
    raise ValueError('target_aoi must be set when using AOI-based modes')
if search_mode in {'region', 'region_in_date_range'} and not str(target_region).strip():
    raise ValueError('target_region must be set when using region-based modes')

target_aoi_norm = _norm(target_aoi)
target_region_norm = _norm(target_region)
records = []

for root in search_roots:
    if not root.exists():
        continue
    for shp in root.glob('**/Shorelines/*.shp'):
        if shp.stem.lower().startswith('[aoierr]'):
            continue
        if len(shp.parts) < 5 or shp.parts[-2].lower() != 'shorelines':
            continue

        region = shp.parts[-4]
        aoi = shp.parts[-3]
        modified = pd.Timestamp(shp.stat().st_mtime, unit='s')
        stem_aoi = shp.stem.rsplit('_', 1)[0]

        matches_aoi = target_aoi_norm in {_norm(aoi), _norm(stem_aoi)}
        matches_region = target_region_norm == _norm(region)
        matches_date = modified > cutoff_date

        include = False
        if search_mode == 'date':
            include = matches_date
        elif search_mode == 'aoi':
            include = matches_aoi
        elif search_mode == 'aoi_in_date_range':
            include = matches_aoi and matches_date
        elif search_mode == 'region':
            include = matches_region
        elif search_mode == 'region_in_date_range':
            include = matches_region and matches_date

        if include:
            records.append({
                'source_root': str(root),
                'region': region,
                'aoi': aoi,
                'shoreline_path': str(shp),
                'modified': modified,
            })

new_shorelines = pd.DataFrame(records, columns=['source_root', 'region', 'aoi', 'shoreline_path', 'modified'])
if not new_shorelines.empty:
    new_shorelines = new_shorelines.sort_values(['region', 'aoi', 'modified']).reset_index(drop=True)

mode_label = {
    'date': f'Date range (modified > {cutoff_date.date()})',
    'aoi': f'AOI only ({target_aoi})',
    'aoi_in_date_range': f'AOI in date range ({target_aoi}; modified > {cutoff_date.date()})',
    'region': f'Region only ({target_region})',
    'region_in_date_range': f'Region in date range ({target_region}; modified > {cutoff_date.date()})',
}[search_mode]

print(f'Mode: {mode_label} | Matches: {len(new_shorelines)}')
new_shorelines

In [ ]:
import ast
from pathlib import Path
from typing import Optional
from difflib import SequenceMatcher

try:
    from rapidfuzz import process as rapidfuzz_process
except ImportError:
    rapidfuzz_process = None

CPS_error_lookup = {1: 0.43, 2: 0.73, 3: 0.97, 4: 2.07, 5: 8.59}


def extract_best_match(query, choices):
    choices = list(choices)
    if len(choices) == 0:
        return None, 0, None
    if rapidfuzz_process is not None:
        return rapidfuzz_process.extractOne(query=query, choices=choices)
    query_text = str(query).lower()
    best_choice = None
    best_score = -1.0
    for choice in choices:
        score = SequenceMatcher(None, query_text, str(choice).lower()).ratio()
        if score > best_score:
            best_choice = choice
            best_score = score
    return best_choice, int(best_score * 100), None


def norm_text(value: str) -> str:
    return ''.join(ch for ch in str(value).lower() if ch.isalnum())


def to_source_relpath(filename: str) -> str:
    text = str(filename).replace('\\', '/')
    for marker in ('MaxarImagery/HighFreq/', 'Retrolens/', 'LDS/', 'Archive/Gabrielle/', 'skyvuw/'):
        if marker in text:
            return text[text.index(marker):]
    if text.startswith('Z:/'):
        return text[3:]
    return text.lstrip('/')


def get_source(filename: str, shapefile: Optional[gpd.GeoDataFrame] = None) -> str:
    # Intentionally path-based: do not depend on shapefile Source attributes.
    rel = to_source_relpath(filename)
    if rel.startswith('Retrolens/') or rel.startswith('RL/') or 'Retrolens/' in rel:
        return 'RL'
    if rel.startswith('MaxarImagery/HighFreq/') or 'MaxarImagery/HighFreq/' in rel:
        return 'MAX'
    if rel.startswith('LDS/') or '/LDS/' in rel:
        return 'LDS'
    return 'Unknown'


def parse_resolution(value):
    if pd.isna(value):
        return pd.NA
    if isinstance(value, (tuple, list, np.ndarray)):
        return float(value[0]) if len(value) else pd.NA
    text = str(value).strip()
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, (tuple, list)) and len(parsed):
            return float(parsed[0])
    except Exception:
        pass
    if ',' in text:
        return float(text.replace('(', '').replace(')', '').split(',')[0].strip())
    try:
        return float(text)
    except Exception:
        return pd.NA


def get_georef_er(scale, taranaki=False):
    # Photoscale thresholds use horizontal-accuracy georef values provided by the spec.
    if pd.isna(scale) or not scale:
        return pd.NA
    scale = float(scale)
    if scale < 20000:
        return 2.09
    elif scale < 30000:
        return 2.43
    return 2.90


def normalize_month_typos(token: str) -> str:
    # Known month typos/variants in historical filenames.
    replacements = {
        'APRL': 'APR',
        'SEPT': 'SEP',
        'JUNE': 'JUN',
        'JULY': 'JUL',
    }
    out = str(token).upper()
    for bad, good in replacements.items():
        out = out.replace(bad, good)
    return out


def parse_date_from_stem(stem: str):
    s = str(stem)

    m = re.search(r'(\d{1,2}[A-Za-z]{3,4}\d{4})', s)
    if m:
        token = normalize_month_typos(m.group(1))
        for fmt in ('%d%b%Y', '%d%B%Y'):
            try:
                return pd.to_datetime(token, format=fmt).date()
            except Exception:
                pass

    m = re.search(r'(\d{4}-\d{2}-\d{2})', s)
    if m:
        try:
            return pd.to_datetime(m.group(1), format='%Y-%m-%d').date()
        except Exception:
            pass

    m = re.search(r'(\d{8})', s)
    if m:
        try:
            return pd.to_datetime(m.group(1), format='%Y%m%d').date()
        except Exception:
            pass

    return None


def parse_dsas_date_from_filename(filename: str):
    dt = parse_date_from_stem(Path(filename).stem)
    if dt is None:
        return None
    return pd.Timestamp(dt).strftime('%d/%m/%Y').lstrip('0')


def get_scale(filename, dsas_date, year):
    path = Path(filename)
    parts = path.parts
    terminator_candidates = [parts.index(name) for name in ('Stack', 'Shorelines') if name in parts]
    if not terminator_candidates:
        raise ValueError(f'Could not find Stack/Shorelines in {filename}')
    terminator = min(terminator_candidates)
    root_dir = Path(*parts[:terminator])
    csv_candidates = list(root_dir.glob('*.csv'))
    if len(csv_candidates) == 0:
        raise ValueError(f'No CSV found for {root_dir}')
    if len(csv_candidates) > 1:
        csv_candidates = [csv_candidates[0]]
    csv_filename = csv_candidates[0]
    try:
        csv = pd.read_csv(csv_filename, encoding='cp1252')
    except UnicodeDecodeError:
        csv = pd.read_excel(csv_filename)
        if 'Date' in csv.columns:
            csv['Date'] = csv['Date'].astype(str)

    matched_date, score, _ = extract_best_match(query=dsas_date, choices=csv.Date.dropna().astype(str).unique())
    if score < 80:
        matched_date, score, _ = extract_best_match(query=year, choices=csv.Date.dropna().astype(str).unique())

    if 'RMSE' in csv.columns:
        filtered = csv[(csv.Date.astype(str) == matched_date) & ~csv.RMSE.isna()]
    else:
        filtered = csv[(csv.Date.astype(str) == matched_date)]

    scales = filtered.Scale.unique() if 'Scale' in filtered.columns else []
    if len(scales) == 0 and 'Scale' in csv.columns:
        filtered = csv[csv.Date.astype(str).str.contains(str(matched_date), na=False)]
        scales = filtered.Scale.unique()
    if len(scales) == 0:
        filtered = csv[csv.Date.astype(str).str.contains(str(year), na=False)]
        scales = filtered.Scale.unique() if 'Scale' in filtered.columns else []
    if len(scales) > 1:
        scales = filtered.Scale.value_counts()
        scales = [scales.index[0]]
    if len(scales) == 0:
        raise ValueError(f"Can't find a scale for {filename}")
    return scales[0]


def infer_stack_dirs(shoreline_path: str):
    p = Path(shoreline_path)
    parts = list(p.parts)
    candidates = []
    if 'Shorelines' in parts:
        i = parts.index('Shorelines')
        aoi_root = Path(*parts[:i])
        candidates.append(aoi_root / 'Stack')
        candidates.append(aoi_root / 'Imagery' / 'Stack')
    else:
        parent = p.parent
        candidates.append(parent / 'Stack')
        candidates.append(parent / 'Imagery' / 'Stack')
    uniq = []
    seen = set()
    for c in candidates:
        key = str(c).lower()
        if key not in seen:
            seen.add(key)
            uniq.append(c)
    return uniq


def pick_stack_raster(shoreline_path: str):
    stem = Path(shoreline_path).stem
    stem_norm = norm_text(stem)
    shoreline_date = parse_date_from_stem(stem)
    year_match = re.search(r'(\d{4})', stem)
    year = year_match.group(1) if year_match else None

    for stack_dir in infer_stack_dirs(shoreline_path):
        if not stack_dir.exists():
            continue

        rasters = []
        for ext in ('*.jp2', '*.JP2', '*.tif', '*.TIF', '*.tiff', '*.TIFF'):
            rasters.extend(stack_dir.glob(ext))
        if len(rasters) == 0:
            continue

        # 1) Strict DSAS-date match first.
        if shoreline_date is not None:
            exact_date = [r for r in rasters if parse_date_from_stem(r.stem) == shoreline_date]
            if len(exact_date) == 1:
                return exact_date[0], 'stack-date-match'
            if len(exact_date) > 1:
                direct = [r for r in exact_date if stem_norm in norm_text(r.stem)]
                if len(direct) == 1:
                    return direct[0], 'stack-date-plus-stem-match'
                return exact_date[0], 'stack-date-multi-first'

        # 2) Direct stem inclusion as second preference.
        direct = [r for r in rasters if stem_norm in norm_text(r.stem)]
        if len(direct) == 1:
            return direct[0], 'stack-direct-match'
        if len(direct) > 1:
            rasters = direct

        # 3) Fuzzy fallback, but only with year agreement and strong score.
        best_name, score, idx = extract_best_match(stem, [r.stem for r in rasters])
        if best_name is None:
            continue
        if idx is not None and idx < len(rasters):
            candidate = rasters[idx]
        else:
            candidate = next((r for r in rasters if r.stem == best_name), rasters[0])

        if year is not None and year not in candidate.stem:
            continue
        if score >= 90:
            return candidate, f'stack-fuzzy-match:{int(score)}'

    return None, 'stack-raster-not-found'


def resolve_pixel_er(filename, shapefile=None, meta_row=None):
    # Pull pixel size from source mosaic metadata in Stack folders (cell size).
    raster_path, match_method = pick_stack_raster(filename)
    if raster_path is None:
        return pd.NA, match_method
    try:
        with rasterio.open(raster_path) as src:
            xres, yres = src.res
            pixel = float(abs(xres))
            return pixel, f'rasterio:{match_method}:{raster_path.name}'
    except Exception as e:
        return pd.NA, f'raster-read-error:{e}'


def resolve_photoscale_and_georef(filename, shapefile=None, meta_row=None, source=None):
    # Derive from source type + external metadata/CSV lookups, not shapefile attributes.
    source = source or get_source(filename, shapefile)

    if source in {'MAX', 'Max', 'max', 'PLE', 'CRI', 'NEO', 'PNE', 'GE1', 'JIN', 'JIL', 'SAT', 'VEX'}:
        return pd.NA, 1.17, 'fixed-by-source'

    if source == 'LDS':
        return pd.NA, 0, 'fixed-by-source'

    if source in {'RL', 'RLN', 'RLS', 'Rl', 'RS'}:
        try:
            dsas_date = parse_dsas_date_from_filename(filename)
            year_match = re.search(r'(\d{4})', str(filename))
            year = year_match.group(1) if year_match else None
            if dsas_date is None or year is None:
                raise ValueError('Could not parse date/year from filename for scale lookup')
            scale = get_scale(filename, dsas_date, year)
            return scale, get_georef_er(scale), 'retrolens-csv-scale'
        except Exception as e:
            print(f'Could not resolve photoscale/georef for {filename}: {e}')
            return pd.NA, pd.NA, 'scale-lookup-failed'

    return pd.NA, pd.NA, 'unknown-source'


def summarize_uncertainty_source(filename, shapefile):
    source = get_source(filename, shapefile)
    pixel_er, pixel_method = resolve_pixel_er(filename, shapefile, None)
    photoscale, georef_er, georef_method = resolve_photoscale_and_georef(filename, shapefile, None, source)
    return {
        'Source': source,
        'Pixel_ER': pixel_er,
        'Photoscale': photoscale,
        'Georef_ER': georef_er,
        'Pixel_ER_method': pixel_method,
        'Georef_ER_method': georef_method,
    }

In [ ]:
if Path('meta.csv').exists():
    meta = pd.read_csv('meta.csv')
    meta['filename'] = meta['filename'].astype(str).str.replace('\\', '/', regex=False)
else:
    meta = pd.DataFrame()

if new_shorelines.empty:
    raise ValueError('No shoreline files matched the current selection.')

uncy_rows = []
missing_rows = []

for rec in tqdm(new_shorelines.to_dict('records'), total=len(new_shorelines)):
    filename = rec['shoreline_path']
    rel_filename = to_source_relpath(filename)
    try:
        shapefile = gpd.read_file(filename)
    except Exception as e:
        missing_rows.append({
            'filename': rel_filename,
            'path': filename,
            'error': f'Could not read shapefile: {e}',
            'write_skipped_due_to_incomplete_components': False,
        })
        continue

    if len(shapefile) == 0:
        missing_rows.append({
            'filename': rel_filename,
            'path': filename,
            'error': 'Empty shapefile',
            'write_skipped_due_to_incomplete_components': False,
        })
        continue

    if 'CPS' not in shapefile.columns:
        missing_rows.append({
            'filename': rel_filename,
            'path': filename,
            'error': 'No CPS column on shoreline shapefile',
            'write_skipped_due_to_incomplete_components': False,
        })
        continue

    source_info = summarize_uncertainty_source(filename, shapefile)
    source = source_info['Source']
    pixel_er = source_info['Pixel_ER']
    photoscale = source_info['Photoscale']
    georef_er = source_info['Georef_ER']
    pixel_method = source_info['Pixel_ER_method']
    georef_method = source_info['Georef_ER_method']

    # Treat unresolved core uncertainty inputs as missing/problem files.
    if pd.isna(pixel_er):
        missing_rows.append({
            'filename': rel_filename,
            'path': filename,
            'error': f'Pixel_ER unresolved ({pixel_method})',
            'write_skipped_due_to_incomplete_components': False,
        })
        continue
    if pd.isna(georef_er):
        missing_rows.append({
            'filename': rel_filename,
            'path': filename,
            'error': f'Georef_ER unresolved ({georef_method})',
            'write_skipped_due_to_incomplete_components': False,
        })
        continue

    shapefile = shapefile.copy()

    # Calculate digitizing error from CPS mapping.
    cps = pd.to_numeric(shapefile['CPS'], errors='coerce')
    ed = cps.map(CPS_error_lookup)

    # Require all rows to have complete components before writing any output.
    calc_valid_mask = ed.notna()
    if not calc_valid_mask.all():
        missing_count = int((~calc_valid_mask).sum())
        missing_rows.append({
            'filename': rel_filename,
            'path': filename,
            'error': f'Incomplete components for Total_UNCY: {missing_count} row(s) have missing/invalid CPS->Dig_ER',
            'write_skipped_due_to_incomplete_components': True,
        })
        continue

    # All rows valid: compute Total_UNCY for every row.
    calc_total_uncy = np.sqrt(
        float(pixel_er) ** 2
        + float(georef_er) ** 2
        + ed.astype(float) ** 2
    )

    # Keep non-Total_UNCY behavior add-only (do not overwrite existing values).
    made_changes = False
    if 'Source' not in shapefile.columns:
        shapefile['Source'] = source
        made_changes = True

    if 'Pixel_ER' in shapefile.columns:
        pixel_col = 'Pixel_ER'
    elif 'Pixel_Er' in shapefile.columns:
        pixel_col = 'Pixel_Er'
    else:
        pixel_col = 'Pixel_ER'
        shapefile[pixel_col] = pixel_er
        made_changes = True

    if 'Photoscale' not in shapefile.columns:
        shapefile['Photoscale'] = photoscale
        made_changes = True

    if 'Georef_ER' not in shapefile.columns:
        shapefile['Georef_ER'] = georef_er
        made_changes = True

    if 'Dig_ER' not in shapefile.columns:
        shapefile['Dig_ER'] = ed
        made_changes = True

    # Always refresh Total_UNCY from current calculations once all components are complete.
    if 'Total_UNCY' not in shapefile.columns:
        shapefile['Total_UNCY'] = pd.NA
    shapefile['Total_UNCY'] = calc_total_uncy
    made_changes = True

    try:
        shapefile.to_file(filename)
    except Exception as e:
        missing_rows.append({
            'filename': rel_filename,
            'path': filename,
            'error': f'Could not write updated shapefile: {e}',
            'write_skipped_due_to_incomplete_components': False,
        })
        continue

    dig_er_unique = sorted(ed.dropna().astype(float).unique().tolist())
    total_uncy_numeric = pd.to_numeric(shapefile['Total_UNCY'], errors='coerce')

    uncy_rows.append({
        'filename': rel_filename,
        'path': filename,
        'region': rec['region'],
        'aoi': rec['aoi'],
        'source': source,
        'rows': len(shapefile),
        'pixel_er': pixel_er,
        'pixel_er_method': pixel_method,
        'photoscale': photoscale,
        'georef_er': georef_er,
        'georef_er_method': georef_method,
        'cps_values': ','.join(map(str, sorted(cps.dropna().astype(int).unique().tolist()))),
        'dig_er_values': ', '.join(f'{v:.2f}' for v in dig_er_unique),
        'total_uncy_mean': float(total_uncy_numeric.mean()) if total_uncy_numeric.notna().any() else pd.NA,
        'valid_uncy_rows': int(calc_valid_mask.sum()),
        'missing_uncy_rows': int((~calc_valid_mask).sum()),
        'shapefile_updated': made_changes,
        'write_skipped_due_to_incomplete_components': False,
    })

uncy_summary = pd.DataFrame(uncy_rows).sort_values(['region', 'aoi', 'filename']).reset_index(drop=True)
if len(missing_rows) == 0:
    uncy_missing = pd.DataFrame(columns=['filename', 'path', 'error', 'write_skipped_due_to_incomplete_components'])
else:
    uncy_missing = pd.DataFrame(missing_rows)

print('Valid uncertainty calculations (summary):')
display(uncy_summary)
print('Missing/problem files:')
display(uncy_missing)

output_dir = DATA_DIR
output_dir.mkdir(parents=True, exist_ok=True)
uncy_summary.to_csv(output_dir / 'new_uncy_summary.csv', index=False)
uncy_missing.to_csv(output_dir / 'new_uncy_missing.csv', index=False)
print(f'Saved {output_dir}/new_uncy_summary.csv and {output_dir}/new_uncy_missing.csv')
print(f'Updated {len(uncy_summary)} shoreline files.')
if len(uncy_missing) > 0:
    print(f'Missing/problem files: {len(uncy_missing)}')
else:
    print('Missing/problem files: 0')